# School Calendar Data (K-12)

## Purpose

Build daily **school calendar features** for each M5 state. Back-to-school is the **2nd largest retail event** after winter holidays, and school session status affects category demand — FOODS shifts when kids eat at home during breaks, HOBBIES surges during summer. The M5 calendar contains no school schedule information, so we construct it from documented state-level patterns.

### Features produced
- Binary flags: `is_school_session`, `is_summer_break`, `is_winter_break`, `is_spring_break`
- `back_to_school_proximity`: 30-day linear decay approaching the first day of school
- `back_to_school_recency`: 14-day linear decay following the first day of school (post-start tail)

In [1]:
# ── Uncomment below when running on Google Colab ────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/Colab Notebooks/m5-forecasting-accuracy/notebooks')

BASE_DIR = '..'
RAW_DIR = f'{BASE_DIR}/data/raw'
CLEAN_DIR = f'{BASE_DIR}/data/clean'
FIG_DIR = f'{BASE_DIR}/figures'

import os
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Project dir: {BASE_DIR}')

Project dir: ..


## Step 0: Setup

In [2]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import calendar as cal_mod
import os
import warnings
warnings.filterwarnings('ignore')

# RAW_DIR defined in mount cell above
# CLEAN_DIR defined in mount cell above

DATE_START = '2011-01-29'
DATE_END = '2016-06-19'

## Step 1: Define school calendar rules

Historical school calendars for 2011–2016 are nearly impossible to find online — school districts purge old calendars from their websites. But the *patterns* are well-documented and stable:

| State | School Start | School End | Winter Break | Spring Break |
|---|---|---|---|---|
| **CA** | ~Aug 24 | ~Jun 12 | Dec 20 – Jan 3 | Week before Easter |
| **TX** | ~Aug 25 | ~Jun 4 | Dec 20 – Jan 3 | Mid-March (fixed week) |
| **WI** | ~Sep 2 | ~Jun 11 | Dec 23 – Jan 2 | Thu–Mon around Easter |

Notice the state differences:
- **CA starts earliest** (late Aug) and ends latest (mid-Jun) — longer school year
- **WI starts latest** (early Sep) — traditional post-Labor-Day start
- **TX spring break is fixed** (mid-March) while CA and WI tie it to Easter (which moves year to year)

### Implementation notes

We need `easter_date()` because both CA and WI anchor spring break to Easter. This is the same algorithm from notebook 01b — we duplicate it here so this notebook is self-contained.

**Why we start from 2010**: Our M5 data starts Jan 29, 2011 — which falls in the *middle* of the 2010–2011 academic year. We need the 2010–2011 school calendar to correctly classify Jan–Jun 2011.

In [3]:
def easter_date(year):
    """Easter Sunday (Anonymous Gregorian algorithm)."""
    a = year % 19
    b, c = divmod(year, 100)
    d, e = divmod(b, 4)
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = divmod(c, 4)
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month, day = divmod(h + l - 7 * m + 114, 31)
    return date(year, month, day + 1)

def build_school_calendar(state_id, year):
    """Build school calendar for one academic year (year -> year+1)."""
    easter = easter_date(year + 1)  # Easter for the spring semester

    if state_id == 'CA':
        school_start = date(year, 8, 24)
        school_end = date(year + 1, 6, 12)
        winter_start = date(year, 12, 20)
        winter_end = date(year + 1, 1, 3)
        spring_start = easter - timedelta(days=7)  # Week before Easter
        spring_end = easter - timedelta(days=1)
    elif state_id == 'TX':
        school_start = date(year, 8, 25)
        school_end = date(year + 1, 6, 4)
        winter_start = date(year, 12, 20)
        winter_end = date(year + 1, 1, 3)
        spring_start = date(year + 1, 3, 11)  # Fixed mid-March
        spring_end = date(year + 1, 3, 18)
    elif state_id == 'WI':
        school_start = date(year, 9, 2)
        school_end = date(year + 1, 6, 11)
        winter_start = date(year, 12, 23)
        winter_end = date(year + 1, 1, 2)
        spring_start = easter - timedelta(days=3)  # Thu before Easter
        spring_end = easter + timedelta(days=1)     # Mon after Easter

    return {
        'state_id': state_id,
        'academic_year': f"{year}-{year+1}",
        'school_start': school_start,
        'school_end': school_end,
        'winter_break_start': winter_start,
        'winter_break_end': winter_end,
        'spring_break_start': spring_start,
        'spring_break_end': spring_end,
    }

calendar_records = []
for state in ['CA', 'TX', 'WI']:
    for year in range(2010, 2016):  # 2010-11 through 2015-16
        record = build_school_calendar(state, year)
        calendar_records.append(record)
        print(f"  {state} {record['academic_year']}: "
              f"start={record['school_start']}, end={record['school_end']}, "
              f"spring break={record['spring_break_start']}-{record['spring_break_end']}")

school_cal = pd.DataFrame(calendar_records)
print(f"\n{len(school_cal)} academic year records")

  CA 2010-2011: start=2010-08-24, end=2011-06-12, spring break=2011-04-17-2011-04-23
  CA 2011-2012: start=2011-08-24, end=2012-06-12, spring break=2012-04-01-2012-04-07
  CA 2012-2013: start=2012-08-24, end=2013-06-12, spring break=2013-03-24-2013-03-30
  CA 2013-2014: start=2013-08-24, end=2014-06-12, spring break=2014-04-13-2014-04-19
  CA 2014-2015: start=2014-08-24, end=2015-06-12, spring break=2015-03-29-2015-04-04
  CA 2015-2016: start=2015-08-24, end=2016-06-12, spring break=2016-03-20-2016-03-26
  TX 2010-2011: start=2010-08-25, end=2011-06-04, spring break=2011-03-11-2011-03-18
  TX 2011-2012: start=2011-08-25, end=2012-06-04, spring break=2012-03-11-2012-03-18
  TX 2012-2013: start=2012-08-25, end=2013-06-04, spring break=2013-03-11-2013-03-18
  TX 2013-2014: start=2013-08-25, end=2014-06-04, spring break=2014-03-11-2014-03-18
  TX 2014-2015: start=2014-08-25, end=2015-06-04, spring break=2015-03-11-2015-03-18
  TX 2015-2016: start=2015-08-25, end=2016-06-04, spring break=20

## Step 2: Classify every day and compute proximity

For every `(state, date)` in the M5 range, we walk through all academic year records for that state and determine the day's status. The classification logic has a **priority order**:

1. **Default**: `is_summer_break = True` — if no academic year claims this date, it must be summer
2. **If date falls within an academic year** (`school_start <= d <= school_end`): it's NOT summer, so we check further
3. **If date is within winter break window**: `is_winter_break = True`, `is_school_break = True`
4. **If date is within spring break window**: `is_spring_break = True`, `is_school_break = True`
5. **Otherwise**: `is_school_session = True` — a normal school day

**Back-to-school proximity**: We also compute how many days until the next `school_start` date. We search across all academic year records and take the smallest non-negative `days_diff`. The proximity formula is `max(0, 1 - days_to_start / 30)` — it peaks at 1.0 on the first day of school and decays over the preceding 30 days.

**Why 30 days, not 14 like SNAP/events?** Back-to-school shopping starts much earlier. Parents buy backpacks and school supplies 3-4 weeks before school starts, not 2-3 days before. The longer window matches this real-world shopping pattern.

**Mirror feature — `back_to_school_recency`**: We also compute days *since* the most recent `school_start`, with a separate 14-day decay. The pre- and post-start dynamics are not symmetric: shopping ramps for ~30 days before school starts (procurement window), then decays over ~2 weeks after start as households complete last-minute purchases and routines settle. Carrying both features avoids the cliff in the original `back_to_school_proximity`, which collapsed from 1.0 on start day to 0.0 the next day, discarding signal from the first two weeks of the school year.

In [4]:
BACK_TO_SCHOOL_WINDOW = 30          # pre-start procurement ramp
POST_SCHOOL_DECAY_WINDOW = 14       # post-start decay (asymmetric: shorter tail)

date_range = pd.date_range(DATE_START, DATE_END, freq='D')
states = ['CA', 'TX', 'WI']

records = []

for state in states:
    state_cal = school_cal[school_cal['state_id'] == state]

    for current_date in date_range:
        d = current_date.date()

        # Defaults: assume summer break unless an academic year claims this date
        is_school_session = False
        is_school_break = False
        is_summer_break = True
        is_winter_break = False
        is_spring_break = False
        days_to_school_start = BACK_TO_SCHOOL_WINDOW
        days_since_school_start = POST_SCHOOL_DECAY_WINDOW

        # Check each academic year record for this state
        for _, cal in state_cal.iterrows():
            s_start = cal['school_start']
            s_end = cal['school_end']
            w_start = cal['winter_break_start']
            w_end = cal['winter_break_end']
            sp_start = cal['spring_break_start']
            sp_end = cal['spring_break_end']

            # Is this date within this academic year?
            if s_start <= d <= s_end:
                is_summer_break = False
                if w_start <= d <= w_end:
                    is_winter_break = True
                    is_school_break = True
                elif sp_start <= d <= sp_end:
                    is_spring_break = True
                    is_school_break = True
                else:
                    is_school_session = True

            # Pre-start: how far until the next school start?
            days_diff = (s_start - d).days
            if 0 <= days_diff < days_to_school_start:
                days_to_school_start = days_diff

            # Post-start: how far since the most recent school start?
            days_after = (d - s_start).days
            if 0 <= days_after < days_since_school_start:
                days_since_school_start = days_after

        # Pre-start proximity: 1.0 on start day, decaying over 30 days before
        days_to_school_start = min(days_to_school_start, BACK_TO_SCHOOL_WINDOW)
        back_to_school_proximity = max(0.0, 1.0 - days_to_school_start / BACK_TO_SCHOOL_WINDOW)

        # Post-start recency: 1.0 on start day, decaying over 14 days after
        days_since_school_start = min(days_since_school_start, POST_SCHOOL_DECAY_WINDOW)
        back_to_school_recency = max(0.0, 1.0 - days_since_school_start / POST_SCHOOL_DECAY_WINDOW)

        records.append({
            'state_id': state,
            'date': current_date,
            'is_school_session': int(is_school_session),
            'is_school_break': int(is_school_break),
            'is_summer_break': int(is_summer_break),
            'is_winter_break': int(is_winter_break),
            'is_spring_break': int(is_spring_break),
            'days_to_school_start': days_to_school_start,
            'back_to_school_proximity': round(back_to_school_proximity, 4),
            'days_since_school_start': days_since_school_start,
            'back_to_school_recency': round(back_to_school_recency, 4),
        })

school_features = pd.DataFrame(records)

print(f"School features shape: {school_features.shape}")
print(f"\nStatus distribution (all states):")
for col in ['is_school_session', 'is_school_break', 'is_summer_break', 'is_winter_break', 'is_spring_break']:
    pct = school_features[col].mean() * 100
    print(f"  {col}: {pct:.1f}% of days")

print(f"\nBack-to-school proximity > 0.5 (pre-start ramp): "
      f"{(school_features['back_to_school_proximity'] > 0.5).mean()*100:.1f}% of days")
print(f"Back-to-school recency > 0.5 (post-start tail):   "
      f"{(school_features['back_to_school_recency']   > 0.5).mean()*100:.1f}% of days")

School features shape: (5907, 11)

Status distribution (all states):
  is_school_session: 74.1% of days
  is_school_break: 5.5% of days
  is_summer_break: 20.4% of days
  is_winter_break: 3.5% of days
  is_spring_break: 2.0% of days

Back-to-school proximity > 0.5 (pre-start ramp): 3.8% of days
Back-to-school recency > 0.5 (post-start tail):   1.8% of days


### Reading the results

The distribution should make intuitive sense:
- **~74% school session** — roughly Sep through Jun minus breaks, so about 9 months / 12 months
- **~20% summer break** — roughly mid-Jun through late Aug, about 2.5 months / 12 months
- **~3.5% winter break** — about 2 weeks per year
- **~2% spring break** — about 1 week per year

These proportions confirm our calendar rules are reasonable. If summer were 50%, that would signal a bug in our start/end dates.

## Step 3: Save output

One file with 9 columns. The binary flags (`is_school_session`, `is_summer_break`, etc.) let the model learn category-specific effects ("HOBBIES sells X% more during summer break"). The `back_to_school_proximity` continuous feature captures the pre-school shopping ramp.

**Merge key for downstream**: `(state_id, date)` — school calendars vary by state.

In [5]:
school_features.to_csv(f'{CLEAN_DIR}/school_calendar.csv', index=False)
print(f"Saved: school_calendar.csv ({len(school_features)} rows)")
print(f"Columns: {list(school_features.columns)}")
print(f"Merge key for downstream: (state_id, date)")

Saved: school_calendar.csv (5907 rows)
Columns: ['state_id', 'date', 'is_school_session', 'is_school_break', 'is_summer_break', 'is_winter_break', 'is_spring_break', 'days_to_school_start', 'back_to_school_proximity', 'days_since_school_start', 'back_to_school_recency']
Merge key for downstream: (state_id, date)


---
## Conclusion

This notebook produces `school_calendar.csv` — 11 columns of school session status and back-to-school proximity per state-day. The calendar reconstructs typical 2010–2016 patterns from documented state norms; specific district variation (e.g., Los Angeles vs. Fresno start dates) is not captured.

**Two proximity features, asymmetric windows**:
- `back_to_school_proximity`: 30-day pre-start ramp (procurement window — parents shop weeks ahead)
- `back_to_school_recency`: 14-day post-start tail (last-minute purchases settle within ~2 weeks)

Carrying both features avoids the cliff at start day and lets the model learn each side of the event independently.

**Merge key for downstream**: `(state_id, date)`.